# Chapter 1 — Why Are We Still Hand-Writing Prompts?

**Book alignment:** DSPy From First Principles, Chapter 1

**Question this notebook isolates:** Do three unrelated rewrite failures (entity violation, interface violation, unevaluated style drift) arrive through one prompt string with no independent detector?


In [ ]:
from pathlib import Path
import sys


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "experiments" / "dspy-from-first-principles" / "common").exists():
            return candidate
    raise RuntimeError(
        "Run this notebook from a checkout containing experiments/dspy-from-first-principles"
    )


REPO_ROOT = find_repo_root(Path.cwd().resolve())
EXP_ROOT = REPO_ROOT / "experiments" / "dspy-from-first-principles"
sys.path.insert(0, str(EXP_ROOT))

from common.data import teaching_cases
from common.metrics import score_editorial_output


## Three failures, one string

A competent handwritten prompt plus a brace-finding parser is the starting unit. The three recorded outputs below are failure shapes from the chapter: each travels the same prompt-to-JSON channel, and each needs a different repair.


In [ ]:
import json
from dataclasses import dataclass


@dataclass
class RewriteRequest:
    sentence: str
    goal: str
    context: str


def build_prompt(request: RewriteRequest) -> str:
    return f"""
You are an expert line editor.

Rewrite the sentence to satisfy the editorial goal.
Preserve meaning, entities, point of view, and the author's voice.
Avoid adding new facts.

Context:
{request.context}

Editorial goal:
{request.goal}

Sentence:
{request.sentence}

Return JSON with:
- rewritten_text: the improved sentence
- rationale: a short explanation of what changed
- confidence: a number from 0.0 to 1.0
""".strip()


def parse_json_object(raw: str) -> dict:
    start = raw.find("{")
    end = raw.rfind("}")
    if start == -1 or end == -1:
        raise ValueError("no JSON object found in response")
    return json.loads(raw[start:end + 1])


cases = {c.case_id: c for c in teaching_cases()}
ed001 = cases["ed-001"]
request = RewriteRequest(sentence=ed001.sentence, goal=ed001.goal, context=ed001.context)
prompt = build_prompt(request)

# Recorded stand-ins for three observed failure shapes. No LM is called.
renamed = {
    "rewritten_text": "Jason opened the door, afraid of what waited inside.",
    "rationale": "Tightened the sentence.",
    "confidence": 0.88,
}
broken_keys = {
    "rewrite": "Jalen opened the door, afraid of what he would find.",
    "why": "Removed the conjunction.",
    "confidence": "high",
}
overwrought = {
    "rewritten_text": "Jalen hurled the door aside, his heart hammering against the cage of his ribs.",
    "rationale": "Heightened the tension.",
    "confidence": 0.94,
}

print(f"prompt characters: {len(prompt)}")
print(f"v1 renamed protagonist: {score_editorial_output(ed001, renamed['rewritten_text']).score:.2f}")
print(f"v1 overwrought rewrite: {score_editorial_output(ed001, overwrought['rewritten_text']).score:.2f}")
print(f"v1 unchanged sentence:  {score_editorial_output(ed001, ed001.sentence).score:.2f}")


In [ ]:
# Failure 1: renamed protagonist trips the deterministic hard gate.
gate = score_editorial_output(ed001, renamed["rewritten_text"])
assert gate.hard_gate_passed is False
assert gate.score == 0.0
assert "missing_required_entity" in gate.failure_categories

# Failure 2: the interface was requested in prose, not enforced. Parsing
# succeeds, then the caller fails two functions away from the cause.
parsed = parse_json_object(json.dumps(broken_keys))
try:
    parsed["rewritten_text"]
    broke_late = False
except KeyError:
    broke_late = True
assert broke_late

# Failure 3: fluent, schema-valid, and wrong — with the highest confidence.
style = score_editorial_output(ed001, overwrought["rewritten_text"])
assert style.hard_gate_passed is True and style.score > 0.5
assert style.semantic_constraints_unchecked == ed001.semantic_constraints
assert renamed["confidence"] == 0.88 and overwrought["confidence"] == 0.94
print("one channel, three unrelated causes: entity gate, interface, unevaluated style")


## One string carries every concern

Task definition, input names, output schema, behavioral framing, model-specific tuning, evaluation criteria, and version identity all live in the same medium. A v6-versus-v7 diff therefore cannot attribute a behavior change.


In [ ]:
CONCERNS = [
    "task definition",
    "input names",
    "output schema",
    "behavioral framing",
    "model-specific tuning",
    "evaluation criteria",
    "version identity",
]
v6 = prompt
v7 = prompt + "\nNever change character names."
print(f"v6 characters: {len(v6)}")
print(f"v7 characters: {len(v7)}")
print(f"concerns fused into the one string: {len(CONCERNS)}")


In [ ]:
assert v7 != v6
# The diff cannot say which concern moved: task, tactic, schema, or tuning
# all changed inside the same artifact, so no variable was isolated.
assert isinstance(v6, str) and isinstance(v7, str)
assert len(CONCERNS) == 7
print("v7-vs-v6: behavior changed, cause unknown - no isolated variable")


## What we earned

The handwritten prompt is a good discovery tool and a poor engineering unit: three failures with nothing in common arrive looking identical, and prompt versions bundle every concern into one uninterpretable diff.

Notebook 02 / Chapter 2 replaces the string with a program-shaped boundary — named inputs, declared behavior, execution strategy, structured outputs — so that one variable can change while the rest stay fixed.
